# Multilayer Perceptron

In [1]:
from analyses.population_analysis import get_all_combined_exploded_spike_counts
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

spike_counts = get_all_combined_exploded_spike_counts(group_name='Zombies', apply_filter=True, apply_binning=True, bin_size=0.2)
spike_counts['NeuronID'].nunique()

322

In [11]:
spike_counts_single_unit = spike_counts[spike_counts['NeuronID'].str.contains('Unit')]
spike_counts_single_unit['NeuronID'].nunique()

65

In [12]:
# Step 0: Restrict to bins 200–600ms (bins 4 to 8)
spike_counts_window = spike_counts_single_unit[
    (spike_counts_single_unit['TimeBinIndex'] >= 1) &
    (spike_counts_single_unit['TimeBinIndex'] <= 8)
].copy()

# Step 1: Keep only neurons with full coverage of 8 bins in this window
bin_counts = (
    spike_counts_window
    .groupby(['TaskField', 'NeuronID'])['TimeBinIndex']
    .nunique()
    .reset_index(name='BinCount')
)

valid_pairs = bin_counts[bin_counts['BinCount'] == 8][['TaskField', 'NeuronID']]
spike_counts_window

,MonkeyGroup,MonkeyName,TaskField,Channel,BaseChannel,EpochStartStop,TimeBinIndex,SpikeCount,Date,Round No.,NeuronID
1,Zombies,69X,1695747327797000,Channel.C_014_Unit 1,Channel.C_014,"(13.49745, 16.10655)",1,12,2023-09-26,1,AMG_2023-09-26_1_Channel.C_014_Unit 1
2,Zombies,69X,1695747327797000,Channel.C_014_Unit 1,Channel.C_014,"(13.49745, 16.10655)",2,20,2023-09-26,1,AMG_2023-09-26_1_Channel.C_014_Unit 1
3,Zombies,69X,1695747327797000,Channel.C_014_Unit 1,Channel.C_014,"(13.49745, 16.10655)",3,9,2023-09-26,1,AMG_2023-09-26_1_Channel.C_014_Unit 1
4,Zombies,69X,1695747327797000,Channel.C_014_Unit 1,Channel.C_014,"(13.49745, 16.10655)",4,9,2023-09-26,1,AMG_2023-09-26_1_Channel.C_014_Unit 1
5,Zombies,69X,1695747327797000,Channel.C_014_Unit 1,Channel.C_014,"(13.49745, 16.10655)",5,7,2023-09-26,1,AMG_2023-09-26_1_Channel.C_014_Unit 1
...,...,...,...,...,...,...,...,...,...,...,...
324248,Zombies,94B,1702937510582000,Channel.C_029_Unit 1,Channel.C_029,"(1741.8872, 1744.195)",4,1,2023-12-18,3,Unknown_2023-12-18_3_Channel.C_029_Unit 1
324249,Zombies,94B,1702937510582000,Channel.C_029_Unit 1,Channel.C_029,"(1741.8872, 1744.195)",5,1,2023-12-18,3,Unknown_2023-12-18_3_Channel.C_029_Unit 1
324250,Zombies,94B,1702937510582000,Channel.C_029_Unit 1,Channel.C_029,"(1741.8872, 1744.195)",6,0,2023-12-18,3,Unknown_2023-12-18_3_Channel.C_029_Unit 1
324251,Zombies,94B,1702937510582000,Channel.C_029_Unit 1,Channel.C_029,"(1741.8872, 1744.195)",7,2,2023-12-18,3,Unknown_2023-12-18_3_Channel.C_029_Unit 1


In [16]:
bin_counts

,TaskField,NeuronID,BinCount
0,1695747327797000,AMG_2023-09-26_1_Channel.C_014_Unit 1,8
1,1695747327797000,AMG_2023-09-26_1_Channel.C_018_Unit 1,8
2,1695747327797000,AMG_2023-09-26_1_Channel.C_018_Unit 2,8
3,1695747327797000,AMG_2023-09-26_1_Channel.C_020_Unit 1,8
4,1695747327797000,AMG_2023-09-26_1_Channel.C_022_Unit 1,8
...,...,...,...
5740,1702937510371000,Unknown_2023-12-18_3_Channel.C_021_Unit 1,8
5741,1702937510371000,Unknown_2023-12-18_3_Channel.C_029_Unit 1,8
5742,1702937510582000,Unknown_2023-12-18_3_Channel.C_015_Unit 1,8
5743,1702937510582000,Unknown_2023-12-18_3_Channel.C_021_Unit 1,8


In [13]:
filtered_df = spike_counts_window.merge(valid_pairs, on=['TaskField', 'NeuronID'])

# Step 2: Create feature names like NeuronID_bin5
filtered_df['Feature'] = (
    filtered_df['NeuronID'] + '_bin' + filtered_df['TimeBinIndex'].astype(str)
)

# Step 3: Pivot
pivot_df = filtered_df.pivot_table(
    index='TaskField',
    columns='Feature',
    values='SpikeCount',
    fill_value=0
)
pivot_df

Feature,AMG_2023-09-26_1_Channel.C_014_Unit 1_bin1,AMG_2023-09-26_1_Channel.C_014_Unit 1_bin2,AMG_2023-09-26_1_Channel.C_014_Unit 1_bin3,AMG_2023-09-26_1_Channel.C_014_Unit 1_bin4,AMG_2023-09-26_1_Channel.C_014_Unit 1_bin5,AMG_2023-09-26_1_Channel.C_014_Unit 1_bin6,AMG_2023-09-26_1_Channel.C_014_Unit 1_bin7,AMG_2023-09-26_1_Channel.C_014_Unit 1_bin8,AMG_2023-09-26_1_Channel.C_018_Unit 1_bin1,AMG_2023-09-26_1_Channel.C_018_Unit 1_bin2,...,Unknown_2023-12-18_3_Channel.C_021_Unit 1_bin7,Unknown_2023-12-18_3_Channel.C_021_Unit 1_bin8,Unknown_2023-12-18_3_Channel.C_029_Unit 1_bin1,Unknown_2023-12-18_3_Channel.C_029_Unit 1_bin2,Unknown_2023-12-18_3_Channel.C_029_Unit 1_bin3,Unknown_2023-12-18_3_Channel.C_029_Unit 1_bin4,Unknown_2023-12-18_3_Channel.C_029_Unit 1_bin5,Unknown_2023-12-18_3_Channel.C_029_Unit 1_bin6,Unknown_2023-12-18_3_Channel.C_029_Unit 1_bin7,Unknown_2023-12-18_3_Channel.C_029_Unit 1_bin8
TaskField,,,,,,,,,,,,,,,,,,,,,
1695747327797000,12.0,20.0,9.0,9.0,7.0,4.0,10.0,3.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1695747328534000,5.0,5.0,10.0,5.0,9.0,1.0,6.0,4.0,3.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1695747328851000,8.0,5.0,13.0,10.0,1.0,6.0,7.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1695747328947000,8.0,7.0,4.0,7.0,5.0,3.0,4.0,7.0,3.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1695747328988000,10.0,9.0,5.0,3.0,8.0,6.0,3.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1702937509158000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
1702937509493000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,2.0,0.0,0.0,1.0,2.0,0.0,1.0,1.0
1702937510011000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0,1.0


In [14]:
# Step 4: Get MonkeyName labels
task_to_label = (
    spike_counts.drop_duplicates(subset=['TaskField'])[['TaskField', 'MonkeyName']]
    .set_index('TaskField')
)

y = task_to_label.loc[pivot_df.index, 'MonkeyName'].values
X = pivot_df.values
feature_names = pivot_df.columns.tolist()

# Optional: check shape
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1148, 520)
y shape: (1148,)


In [15]:
# Step 1: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 2: Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
from sklearn.decomposition import PCA
pca = PCA(n_components=50)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Step 3: Train the MLPClassifier
mlp = MLPClassifier(
    hidden_layer_sizes=(1000,),
    activation='relu',
    solver='adam',
    alpha=0.01,
    max_iter=10000
)
mlp.fit(X_train_pca, y_train)

# Step 4: Evaluate
y_pred = mlp.predict(X_test_pca)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

        110E       0.14      0.16      0.15        25
        143H       0.17      0.12      0.14        26
        151J       0.05      0.04      0.04        25
         67G       0.12      0.16      0.14        25
         69X       0.07      0.08      0.07        26
        7124       0.15      0.15      0.15        26
         72X       0.09      0.12      0.10        26
         87J       0.05      0.04      0.04        26
         94B       0.06      0.04      0.05        25

    accuracy                           0.10       230
   macro avg       0.10      0.10      0.10       230
weighted avg       0.10      0.10      0.10       230

Confusion Matrix:
[[4 2 2 4 4 2 3 1 3]
 [2 3 2 1 4 4 2 5 3]
 [4 1 1 6 3 1 4 4 1]
 [3 0 1 4 4 4 3 1 5]
 [3 2 3 4 2 3 2 5 2]
 [5 0 6 2 3 4 4 2 0]
 [2 2 4 3 6 4 3 1 1]
 [2 5 2 2 4 3 5 1 2]
 [4 3 1 6 0 2 6 2 1]]
